In [86]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd


In [87]:
class vocab:
    def __init__(self):
        self.word2idx={"<pad>":0,"<sos>":1,"<eos>":2}
        self.idx2word={0:"<pad>",1:"<sos>",2:"<eos>"}
        self.count=3

    def add_sentence(self,sentence):
        for word in sentence.split():
            if word not in self.word2idx:
                self.word2idx[word]=self.count
                self.idx2word[self.count]=word
                self.count +=1

In [88]:
source_vocab=vocab()
target_vocab=vocab()

In [89]:
source_sentence="i eat"
target_sentence="je mange"

In [90]:
source_vocab.add_sentence(source_sentence)
target_vocab.add_sentence(target_sentence)

In [91]:
source_indices = [source_vocab.word2idx[w] for w in source_sentence.split()]
target_indices = [target_vocab.word2idx["<sos>"]] + \
              [target_vocab.word2idx[w] for w in target_sentence.split()] + \
              [target_vocab.word2idx["<eos>"]]

In [92]:
source_tensor = torch.tensor(source_indices).unsqueeze(1)  # (seq_len, batch)
target_tensor = torch.tensor(target_indices).unsqueeze(1)


In [93]:
class Encoder(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_dim):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,embedding_dim)
        self.rnn=nn.RNN(embedding_dim,hidden_dim)
    def forward(self,x):
        embedded= self.embedding(x)
        outputs,hidden=self.rnn(embedded)
        return hidden                                


In [94]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        embedded = self.embedding(x)
        outputs, hidden = self.rnn(embedded, hidden)
        predictions = self.fc(outputs)
        return predictions, hidden


In [95]:
embedding_dim = 4
hidden_dim=4
encoder = Encoder(len(source_vocab.word2idx), embedding_dim , hidden_dim)
decoder = Decoder(len(target_vocab.word2idx), embedding_dim , hidden_dim)


In [96]:
# Encoder forward
context = encoder(source_tensor)

# Decoder forward (teacher forcing)
decoder_input = target_tensor[:-1]        # remove <eos>
decoder_target = target_tensor[1:]        # shifted target

decoder_output, _ = decoder(decoder_input, context)


In [97]:
criterion = nn.CrossEntropyLoss()

loss = criterion(
    decoder_output.view(-1, decoder_output.size(-1)),
    decoder_target.view(-1)
)

print("Loss:", loss.item())



Loss: 1.5645180940628052


In [98]:
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.01)

optimizer.zero_grad()
loss.backward()
optimizer.step()
print("Loss updated (after one step):", loss.item())

Loss updated (after one step): 1.5645180940628052


In [100]:
EPOCHS = 100

for epoch in range(EPOCHS):
    optimizer.zero_grad()
    context = encoder(source_tensor)
    decoder_input = target_tensor[:-1]
    decoder_target = target_tensor[1:]
    decoder_output, _ = decoder(decoder_input, context)
    loss = criterion(decoder_output.view(-1, decoder_output.size(-1)), decoder_target.view(-1))
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1} Loss: {loss.item():.4f}")


Epoch 10 Loss: 1.1180
Epoch 20 Loss: 0.7864
Epoch 30 Loss: 0.5150
Epoch 40 Loss: 0.3343
Epoch 50 Loss: 0.2269
Epoch 60 Loss: 0.1647
Epoch 70 Loss: 0.1269
Epoch 80 Loss: 0.1023
Epoch 90 Loss: 0.0852
Epoch 100 Loss: 0.0727


In [101]:
with torch.no_grad():
    context = encoder(source_tensor)
    decoder_input = torch.tensor([target_vocab.word2idx["<sos>"]]).unsqueeze(0)
    hidden = context
    predicted_sentence = []
    max_len = 10
    
    for _ in range(max_len):
        decoder_output, hidden = decoder(decoder_input, hidden)
        predicted_idx = decoder_output.argmax(dim=2).item()
        
        if predicted_idx == target_vocab.word2idx["<eos>"]:
            break
        
        predicted_sentence.append(target_vocab.idx2word[predicted_idx])
        decoder_input = torch.tensor([predicted_idx]).unsqueeze(0)

print("Predicted Translation:", " ".join(predicted_sentence))

Predicted Translation: je mange


In [102]:
import pandas as pd

In [105]:
data=pd.read_csv("TR2EN.txt",sep="\t",header=None,names=["source", "target"])

In [107]:
source_vocab = vocab()
target_vocab = vocab()

for _, row in data.iterrows():
    source_vocab.add_sentence(row["source"])
    target_vocab.add_sentence(row["target"])

In [145]:
pairs = []

for _, row in data.iterrows():
    source_encoded = [source_vocab.word2idx[w] for w in row["source"].split()]
    target_encoded = (
        [target_vocab.word2idx["<sos>"]] +
        [target_vocab.word2idx[w] for w in row["target"].split()] +
        [target_vocab.word2idx["<eos>"]]
    )

    pairs.append((
        torch.tensor(source_encoded),
        torch.tensor(target_encoded)
    ))


In [146]:
encoder = Encoder(
    vocab_size=source_vocab.count,
    embed_dim=64,
    hidden_dim=128
)

decoder = Decoder(
    vocab_size=target_vocab.count,
    embed_dim=64,
    hidden_dim=128
)


In [147]:
def collate_fn(batch):
    source_batch, target_batch = zip(*batch)

    source_batch = torch.nn.utils.rnn.pad_sequence(
        source_batch, batch_first=True, padding_value=0
    )

    target_batch = torch.nn.utils.rnn.pad_sequence(
        target_batch, batch_first=True, padding_value=0
    )

    return source_batch, target_batch


In [148]:
from torch.utils.data import DataLoader 
loader = DataLoader(
    pairs,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

In [149]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)              # (B, S, E)
        outputs, hidden = self.rnn(embedded)      # hidden: (1, B, H)
        return hidden


In [150]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded, hidden)
        prediction = self.fc(output)
        return prediction, hidden


In [152]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()), lr=0.001
)




In [153]:
EPOCHS=10
for epoch in range(EPOCHS):
    total_loss = 0

    for source, target in loader:
        optimizer.zero_grad()

        context = encoder(source)

        decoder_input = target[:, :-1]   # remove <eos>
        decoder_target = target[:, 1:]   # remove <sos>

        output, _ = decoder(decoder_input, context)

        loss = criterion(
            output.reshape(-1, output.size(-1)),
            decoder_target.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

KeyboardInterrupt: 

In [ ]:
with torch.no_grad():
    context = encoder(source_tensor)
    decoder_input = torch.tensor([[target_vocab.word2idx["<sos>"]]])
    hidden = context
    result = []

    for _ in range(20):
        output, hidden = decoder(decoder_input, hidden)
        pred = output.argmax(-1).item()

        if pred == target_vocab.word2idx["<eos>"]:
            break

        result.append(target_vocab.idx2word[pred])
        decoder_input = torch.tensor([[pred]])

print("Translation:", " ".join(result))


IndexError: index out of range in self